# habitable candidates of the entire Exo-MerCat census
- paginated tabularray `longtblr` (`rowhead=1`); marker legend lives with the short table
- exports to `report/figures/rocky_temperate_candidates_table_full.tex`

In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys, pathlib

ROOT = pathlib.Path.cwd()
while not (ROOT / 'crossmatching').exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from report_plots import common, data, tables


In [ ]:
cme, input_table, matched = data.emc_crossmatch()
merger = data.build_param_filler()

e_emc = data.enrich(cme.catalog_table, cme, merger)
data.compute_is_rocky_temperate(e_emc)
len(e_emc)

In [ ]:
tex = tables.build_rocky_temperate_table_tex(
    e_emc,
    common.FIGURES_DIR / "rocky_temperate_candidates_table_full.tex",
    caption=r"Rocky, temperate planet candidates in Exo-MerCat (see \cref{fig:rocky_temp_yield})",
    label="tab:rocky_temp_table_full",
    long=True,
    sma_col_width="3cm",
    include_legend=False,
)
print(tex)


In [ ]:
from datetime import date

# Keep the candidate definition identical between the enriched and published exports.
published = data.enrich(cme.catalog_table, cme, merger, disable_calculations=True)
candidate_mask = e_emc["rocky_temp_status"] != ""
candidate_rows = [idx for idx, is_candidate in enumerate(candidate_mask) if is_candidate]

enriched_candidates = e_emc[candidate_rows]
published_candidates = published[candidate_rows]

export_dir = common.ROOT / "report" / "data_exports"
export_dir.mkdir(parents=True, exist_ok=True)
export_date = date.today().isoformat()

enriched_candidates.write(
    export_dir / f"{export_date}-Exo-MerCat-habitable-enriched.csv",
    format="csv",
    overwrite=True,
)
published_candidates.write(
    export_dir / f"{export_date}-Exo-MerCat-habitable-published.csv",
    format="csv",
    overwrite=True,
)
e_emc.write(
    export_dir / f"{export_date}-Exo-MerCat-enriched.csv",
    format="csv",
    overwrite=True,
)
published.write(
    export_dir / f"{export_date}-Exo-MerCat-published.csv",
    format="csv",
    overwrite=True,
)